In [ ]:
# Locate this checkout when launched from the repository or notebook folder.
from pathlib import Path
import sys
_REPOSITORY_ROOT = next(
    p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "aerosandbox/__init__.py").is_file()
    and (p / "studies/TSher_Improvements").is_dir()
)
_STUDY_DIRECTORY = _REPOSITORY_ROOT / "studies/TSher_Improvements/aerodynamics"
for _path in [_REPOSITORY_ROOT, _STUDY_DIRECTORY]:
    if str(_path) not in sys.path:
        sys.path.insert(0, str(_path))


In [ ]:
try:
  import aerosandbox as asb
  import aerosandbox.numpy as np # differentiable numpy
except:
  !pip install aerosandbox

# Example: minimum drag for a given lift

Adapted [from](https://github.com/peterdsharpe/AeroSandbox/blob/master/tutorial/02%20-%20Design%20Optimization/01%20-%20Wing%20Drag%20Minimization%2C%20a%20cautionary%20tale.ipynb)

Throughout the next three notebooks, we are working towards a reasonable model of an RC motor glider, similar to the FAI F5K class. There are many existing examples to reference, such as the [Armsoar Fury](https://www.armsoar.com/collections/electric-gliders/products/fury-2-piece-wing-receiver-ready-assembled?variant=44387211477292), pictured below.

![Armsoar Fury](https://www.armsoar.com/cdn/shop/products/rr_fury_fgrwhf.png?v=1653422556)

Armsoar Fury specs:
- Wingspan: 1490mm
- Wing area: 21.66 dm²
- Wing airfoil: modified Synergy 2
- Horizontal tail area: 2.43 dm²
- Light AUW: ~265 grams
- Standard AUW: ~275 grams
- Strong AUW: ~285 grams
- Min F5K wing loading (12dm²): 289 grams

We will be obeying the $1.5\ \mathrm{m}$ span limit of the F5K class and eventually sizing the propulsion system for zoom climbs, typical of the start of F5K contests. For now, this helps us with a mass estimate.


Given (constants):
- air density $\rho\  [\mathrm{kg/m^3}]$
- dynamic viscosity $\mu\ [\mathrm{\frac{kg}{m\ s}}]$
- aircraft weight $W\ [\mathrm{N}]$

Find (design variables):
- wing aspect ratio $AR\ []$
- wing area $S\ [\mathrm{m^2}]$
- airspeed $u_\infty\ [\mathrm{m/s}]$

In order to (objective):
- minimize drag $D\ [\mathrm{N}]$

Such that (constraints):
- lift = weight (steady flight)
- the design is aerodynamically feasible
- the design is structurally feasible


Since this is an aerodynamics-only analysis, we need to make assumptions about the structures/weight.

Furthermore, we assume that the wing drag dominates and therefore do not model the fuselage or tail at all.

## Aside on viscosity
Viscosity is the fluid analog of structural stiffness.

### Springs

Springs follow the relation $F=kx$ where $F$ is a force, $k$ is a stiffness with units of $[\mathrm{Force/Displacement}]$ and $x$ is a displacement.

$k$ is a material property.

**For a spring, constant force implies constant displacement**


### Structural Mechanics

In structural mechanics, we define $\sigma = E ɛ$, where
- $\sigma\ [\mathrm{Pa}]$, the material *stress*, has units of $[\mathrm{Force/Area}]$
- $E\ [\mathrm{Pa}]$, the material *stiffness*, also has units of $[\mathrm{Force/Area}]$
- $ɛ = \Delta L / L$, the material *strain* is dimensionless and is the fraction of material elongation.

$E$ is a material property.

Materials begin to fail when the stress $\sigma$ exceeds the *yield stress* $\sigma_y$.

**In structural mechanics, constant stress implies constant strain** (same as a spring)

### Fluid Dynamics
In fluid dynamics, we (simplified in 1D) define $\tau = \mu ɛ$, where
- $\tau\ [\mathrm{Pa}]$, the fluid *shear stress*, has units of $[\mathrm{Force/Area}]$
- $\mu\ [\mathrm{Pa \cdot s}]$, the fluid *viscosity*, has units of $[\mathrm{Force/(Area\cdot Time)}]$
- $ɛ = \frac{d \gamma}{dt} [1/\mathrm{s}]$, the material *strain rate*, has units of $[1/\mathrm{Time}]$ and is the rate of fluid shear strain (angular deformation) over time.

$\mu$ is a material property.

**In fluid dynamics, constant stress implies constant *rate of strain***. Instead of constant stress (force) causing constant displacement, it causes constant motion.

### Fluid rotation and deformation:
![normal and shear strain](https://drive.google.com/uc?export=view&id=13bgrWQPl9AzFKGRz-sptJ8PCiT_Z5IEq)

In [ ]:
# constants
rho = 1.225  # [kg/m3], air density
mu = 1.81e-5  # [kg/m.s] == [Pa.s], air dynamic viscosity
g = 9.81  # [m/s2], gravitational acc
W = 0.275 * g  # [N], structural weight (to be lifted) of 275 grams, from the Fury standard layup

Any two of four variables could fully define the wing for our purposes:
- $b\ \mathrm{[m]}$: wing span, from tip to tip
- $c\ \mathrm{[m]}$: wing chord, from leading edge (LE) to trailing edge (TE). We typically consider the mean chord $\bar{c}$.
- $S \ [\mathrm{m^2}]$: wing area, $S=bc$ for a rectangular wing
- $AR\ []$: wing aspect ratio, $AR≡b^2/S$, equivalent to $AR=b/c$ for a rectangular wing

As shown by [Martins and Ning](https://mdobook.github.io/html/build/wing_opt_contours-8461f028ba9de30c3cb643fff3abffff.webp), $AR$ and $S$ provide improved orthogonality when compared to $b$ and $c$, so they are selected. You could easily experiment with any combination, though. This is a similar effect to the log transform explored in the previous notebook.
![mdobook](https://mdobook.github.io/html/build/wing_opt_contours-8461f028ba9de30c3cb643fff3abffff.webp)

In [ ]:
# optimization problem
opti = asb.Opti()

# design variables
# aspect ratio === b^2/S []
# b === wing span [m]
# S === wing area [m2]
AR = opti.variable(init_guess=12, log_transform=True)  # [], aspect ratio
S = opti.variable(init_guess=1, log_transform=True)  # [m2], wing area

# note: why AR and S and not b and cbar? or AR and cbar...
# parametrization. Peter's page has some notes on this too
# mdobook pg 11, fig 1.11 https://flowlab.groups.et.byu.net/mdobook.pdf

airspeed = opti.variable(init_guess=30, lower_bound=0)  # [m/s]

## Dependent variables

### Geometry
Since our later analysis will require $b$ and $c$, we simply compute them according to their definitions from before.

### Drag
For an aircraft, drag is defined as

$D\ \mathrm{[N]} = \frac{1}{2} \rho V^2 S C_D$

Where $q=\frac{1}{2} \rho V^2\ [\mathrm{Pa}]$ is called dynamic pressure, since it has pressure units and is related to the velocity. Pressure is the energy per unit volume in the flow (check the units). Static pressure ($p$) comes from the random motion of fluid particles due to their internal energy (temperature). Dynamic pressure comes from the mean velocity ($V$) of a fluid in motion.

$C_D$ is the drag coefficient and is strongly dependent on the shape and weakly on the Reynolds number.

Note that the lift equation takes the same form as drag, only with $L$ and $C_L$. In automotive aerodynamics, the same equations are used, except downforce is the additive inverse of lift, and the relevant area is taken as the frontal area.

In this part of the code, we apply the $L=W$ constraint for steady (non accelerating) flight, without even having to use an optimizer constraint!

#### Reynolds number
$Re = \frac{\rho V l}{\mu}$, where $l$ is a relevant length scale (in this case, the chord, for development of the boundary layer).

The Reynolds number is the ratio of inertial to viscous forces. The lift and drag equations above are for inertial forces (with $\rho$, which has $[\mathrm{kg}]$ in it).

**Group Exercise: find an equation for a viscous force and derive the Reynolds number formula. Explain why we use the inertial formulas for fluid forces.**

![CD](https://drive.google.com/uc?export=view&id=1BUzOBVrP1jeefNWYs4K-Zmf87rFcOYzO)

If you thought that was interesting, you may be interested in the Buckingham Pi Theorem and a story about the atomic bomb.

#### Drag coefficient

In the early stages of aircraft design, we model the drag coefficient as a quadratic:
$C_D = C_{D_{\mathrm{min}}} + \frac{(C_L-C_{L_{\mathrm{min\ D}}})^2}{\pi AR e_0}$

For the purposes of this example, we will model the viscous (skin friction, two dimensional) component of drag separately than the inviscid (3D, wake energy losses) component, to build more intuition about these sources of drag. We also make the assumption that span efficiency $e=1$ is perfect, which is a good assumption for a sailplane. We also assume that $C_{L_{\mathrm{min\ D}}}=0$, which is a less good assumption.

##### **Viscous Drag**
The drag-per-unit-span (2D drag) $D' [\mathrm{N/m}]$ is expressed in terms of the wall shear stress $\tau_w(x)$, where $x$ is the distance aft of the wing leading edge. We multiply by two to get the top and bottom of the wing.

$D' = 2 \int_0^c \tau_w(x)dx$

Substituting in the definiton of $\tau_w(x) = \frac{1}{2}\rho V^2 c_f(x)$ in terms of the skin friction coefficient $c_f(x)$, we get:

$D' = 2 (\frac{1}{2}\rho V^2) \int_0^c cf(x)dx$

Next, we model $c_f(x) = 0.0592 Re_x^{-1/5}$ for a turbulent boundary layer on a flat plate. This is experimentally determined. In reality, a sailplane like this will have a laminar (thinner, lower drag) boundary layer over some of the chord, before it transitions to turbulent. Since we are neglecting the fuselage and tail, we attempt to compensate by assuming the whole wing is turbulent. Furthermore, this model is for a flat plate producing no lift. The skin friction increases when lift is generated.

Note that $Re_x$ is simply the Reynolds number where $x$ is the reference length.

We can reframe the problem to find the drag coefficient $C_D$. We normalize by $c$ instead of $S$ since we are dealing with $D'$

$C_D = \frac{D'}{\frac{1}{2}\rho V^2 c} = \frac{2}{c} \int_0^c c_f(x)dx$

Integrating,

$C_D = \frac{2}{c} \int_0^c 0.0592 \frac{\rho V x}{\mu}^{-1/5}dx=0.148 \frac{\rho V c}{\mu}^{-1/5} = 0.148 Re_c^{-1/5}$

We call this our viscous drag coefficient,
$C_{D_v} = 0.148 Re_c^{-1/5}$

##### **Inviscid Drag**
Inviscid drag can be modeled with the results of Prandtl's Lifting Line Theory. By modeling a wing with a single horseshoe vortex, Prandtl showed that a wing of infinite span would have no drag other than viscous drag, but that a wing of finite span would lose some energy by accelerating air downwards over its surface.
![Prandtl's Lifting Line](https://eaglepubs.erau.edu/app/uploads/sites/4/2024/12/Glauert_wings_2-1024x681.png)

Prandtl showed that induced drag is a function of $q$, efficiency $e$, which is how similar the lift distribution along the span is to an elliptical shape (minimum induced drag for a fixed span), and the span loading $L/b$.

$D_i = \frac{1}{q \pi e} (\frac{L}{b})^2$

By dividing each side by $qS$, the coefficient can be written.

$C_{D_i} = \frac{C_L^2}{\pi AR e}$

For a fixed amount of lift (vehicle weight), a wider span is more efficient, since along each slice of the wing, it has to turn (accelerate) the air less. This is a general principle in aerodynamics and propulsion. **It is generally more efficient to move a lot of air a little bit than to move a little bit of air a lot.** This comes directly from $F=ma$, which is equivalent to $F=\dot{m} \Delta V$.

Mass flow rate $\dot{m}=\rho V A$.

If force (lift) is increased by increasing $\Delta V$, dynamic pressure goes with $V^2$ and so does the drag force. If force (lift) is increased by increasing $A$, the drag force also scales with $A$. The trick is to make the area bigger without increasing the vehicle weight.

In [ ]:
# dependent variables

# recall AR = b^2/S
# b = sqrt(S*AR)
b = np.sqrt(S * AR)  # [m], wing span
c = S / b  # [m], chord of a rectangular wing (for which S=bc)

# dynamic pressure q = 0.5 * rho * V^2
q = 0.5 * rho * airspeed**2  # [Pa], dynamic pressure
L = W  # [N], lift. L = W for steady flight
CL = L / (q * S)  # [], lift coefficient definition

Re = rho * airspeed * c / mu  # [], Reynolds number definition

# turbulent skin friction coefficient
# cf_turb = 0.0592 Re_x^(-1/5), where Re_x is Re based on distance from LE (BL start)
# plugging in Re_x = rho*airspeed*x/mu from 0 to c dx, we get
# cd_turb = 0.074 Re_c^(-1/5), where Re_c is our Re
# multiply by 2 for 2 sides (top and bottom)
# [] viscous (boundary layer / skin friction / 2D) drag coefficient
CD_v = 0.148 * Re ** (-1 / 5)

# induced drag assuming a span efficiency of 1
# [] inviscid (induced / 3D due to lift) drag coefficient
CD_i = CL**2 / (np.pi * AR)
CD = CD_v + CD_i
D = q * S * CD  # [N], dimensional drag

opti.minimize(D)  # minimum drag for our L = W
# note that we didn't even have to make a constraint for L=W, since it is a definition

# note - why not minimize CD? CDA? aside about Rivians and smart cars

sol = opti.solve()

This is Ipopt version 3.14.11, running with linear solver MUMPS 5.4.1.

Number of nonzeros in equality constraint Jacobian...:        0
Number of nonzeros in inequality constraint Jacobian.:        1
Number of nonzeros in Lagrangian Hessian.............:        6

Total number of variables............................:        3
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Total number of equality constraints.................:        0
Total number of inequality constraints...............:        1
        inequality constraints with only lower bounds:        1
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:        0

iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
   0  5.7284779e+00 0.00e+00 5.15e+00   0.0 0.00e+00    -  0.00e+00 0.00e+00 

In [ ]:
AR_opt = sol(AR)
S_opt = sol(S)
b_opt = sol(b)
D_opt = sol(D)

print(f"AR* = {AR_opt} \nS* = {S_opt} m2 \nb* = {b_opt} m \nD* = {D_opt} N")

AR* = 4.422035084195444e+19 
S* = 1.597755844600741e-11 m2 
b* = 26580.692994733603 m 
D* = 1.7912538697181434e-09 N


In [ ]:
# without the stall constraint, we have infinite AR and no area
# investigate if the CL is reasonable
CL_opt = sol(CL)
print(f"CL* = {CL_opt}")

CL* = 52578687162.780426


After all that, the results are nonsense!
Must ensure that the design is "aerodynamically and structurally feasible". Let's look at aerodynamics first.

A constraint on CL (or an upper bound) is in order.
![stall](https://eaglepubs.erau.edu/app/uploads/sites/4/2021/09/Wing_lift_flaps.png)

**Exercise: add a CL constraint**

In [ ]:
# try adding a stall constraint
# common airfoils stall at 1.5 at higher Re
# in 3D for our low Re, 0.9 is typical

#####
# constrain here
opti.subject_to(CL < 0.9)
#####
sol2 = opti.solve()

This is Ipopt version 3.14.11, running with linear solver MUMPS 5.4.1.

Number of nonzeros in equality constraint Jacobian...:        0
Number of nonzeros in inequality constraint Jacobian.:        3
Number of nonzeros in Lagrangian Hessian.............:        6

Total number of variables............................:        3
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Total number of equality constraints.................:        0
Total number of inequality constraints...............:        2
        inequality constraints with only lower bounds:        1
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:        1

iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
   0  5.7284779e+00 0.00e+00 5.15e+00   0.0 0.00e+00    -  0.00e+00 0.00e+00 

In [ ]:
AR_opt = sol2(AR)
S_opt = sol2(S)
b_opt = sol2(b)
D_opt = sol2(D)

print(f"AR* = {AR_opt} \nS* = {S_opt} m2 \nb* = {b_opt} m \nD* = {D_opt} N")

CL_opt = sol2(CL)
print(f"CL* = {CL_opt}")

AR* = 117.27604732280295 
S* = 1.485998303398922 m2 
b* = 13.201212343986324 m 
D* = 0.07248993450296394 N
CL* = 0.9000000101215802


The output now meets the "aerodynamic feasibility" criteria. The wing span is over the F5K limit of $1.5\  \mathrm{m}$, let's constrain $b$ and $AR$ to reasonable numbers.

In [ ]:
# the result is now somewhat reasonable, except AR=117. is not
# let's constrain the span to 1.5 m, typical of a hand launched glider
# and let's constrain AR <= 10 to maintain some structure in the wing

########
# constrain here
opti.subject_to([b < 1.5, AR < 10])
########

sol3 = opti.solve()

This is Ipopt version 3.14.11, running with linear solver MUMPS 5.4.1.

Number of nonzeros in equality constraint Jacobian...:        0
Number of nonzeros in inequality constraint Jacobian.:        8
Number of nonzeros in Lagrangian Hessian.............:        6

Total number of variables............................:        3
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Total number of equality constraints.................:        0
Total number of inequality constraints...............:        5
        inequality constraints with only lower bounds:        1
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:        4

iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
   0  5.7284779e+00 2.00e+00 4.65e+00   0.0 0.00e+00    -  0.00e+00 0.00e+00 

In [ ]:
AR_opt = sol3(AR)
S_opt = sol3(S)
b_opt = sol3(b)
D_opt = sol3(D)

print(f"AR* = {AR_opt} \nS* = {S_opt} m2 \nb* = {b_opt} m \nD* = {D_opt} N")

CL_opt = sol3(CL)
print(f"CL* = {CL_opt}")

CD_opt = sol3(CD)
print(f"CD* = {CD_opt}")

print(f"L/D * = {CL_opt/CD_opt}")

CD_i_opt = sol3(CD_i)
CD_v_opt = sol3(CD_v)
print(f"CDi* = {CD_i_opt} \nCDv* = {CD_v_opt}")

V_opt = sol3(airspeed)
print(f"V* = {V_opt} m/s")

Re_opt = sol3(Re)
print(f"Re* = {Re_opt}")
# in reality, at a high CL like 0.6 or 0.9, the airfoil CDv is more like 0.04. it isn't a flat plate :)
# this extra drag comes as a function of 2D lift and pressure gradients affecting the boundary layer

AR* = 10.000000098095231 
S* = 0.14083199984061642 m2 
b* = 1.186726595396392 m 
D* = 0.12468735427171908 N
CL* = 0.6877957456355902
CD* = 0.031789243555795016
L/D * = 21.63611551273319
CDi* = 0.015058062430001608 
CDv* = 0.016731181125793408
V* = 6.743220922085857 m/s
Re* = 54159.615592582


From these very crude assumptions, we reach a design very similar to a modern DLG, such as the SuperGee II
https://charlesriverrc.org/articles/on-line-plans/mark-drela-designs/supergee-ii/files/supergee2.pdf
or the Armsoar Fury https://www.armsoar.com/collections/electric-gliders/products/fury-2-piece-wing-receiver-ready-assembled?variant=44387211477292

A peak L/D of ~20 is reasonable, as is our wing area, CDi, and Re.
By assuming the wing was turbulent and ignoring everything else, we got a reasonable estimate of the drag on the total aircraft, on which the wing would really be mostly laminar.

L/D source https://www.rcgroups.com/forums/showatt.php?attachmentid=6115883&d=1379627823. L/D in figure 12 and A5 and Also nicely validates minimum drag speed in Figure A6

You may be thinking: but we need to know the weight, AR limit, and span limit in advance! This only worked so well because you already knew the answer!
Few rebuttals:
- yes, it is generally good practice to calibrate these models to existing aircraft. In this case that happened by accident, which is nice. More work could be done to better match existing designs.
- There is rarely a new aircraft which has nothing in common with existing designs, so leveraging "actuals" is one of the best ways to calibrate design tools
- Now that we have the tool, we can do sensitivity analysis to understand the design space
- By creating structures and propulsion models, we can include weight, strength, and stiffness prediction in our model so they don't have to be known in advance

Sensitivity analysis example: weight growth

In [ ]:
## new variable for sensitivity!
W_sens = opti.variable(init_guess=0)  # [N] dummy variable
W = W + W_sens  # [N]

# dummy variable cannot change
W_sens_con = W_sens == 0
opti.subject_to(W_sens_con)

## code from before

# dependent variables

# recall AR = b^2/S
# b = sqrt(S*AR)
b = np.sqrt(S * AR)  # [m], wing span
c = S / b  # [m], chord of a rectangular wing (for which S=bc)

# dynamic pressure q = 0.5 * rho * V^2
q = 0.5 * rho * airspeed**2  # [Pa], dynamic pressure
L = W  # [N], lift. L = W for steady level flight
CL = L / (q * S)  # [], lift coefficient definition

Re = rho * airspeed * c / mu  # [], Reynolds number definition

# turbulent skin friction coefficient
# cf_turb = 0.0592 Re_x^(-1/5), where Re_x is Re based on distance from LE (BL start)
# plugging in Re_x = rho*airspeed*x/mu from 0 to c dx, we get
# cd_turb = 0.074 Re_c^(-1/5), where Re_c is our Re
# multiply by 2 for 2 sides (top and bottom)
# [] viscous (boundary layer / skin friction / 2D) drag coefficient
CD_v = 0.148 * Re ** (-1 / 5)

# induced drag assuming a span efficiency of 1
# [] inviscid (induced / 3D due to lift) drag coefficient
CD_i = CL**2 / (np.pi * AR)
CD = CD_v + CD_i
D = q * S * CD  # [N], dimensional drag


opti.minimize(D)  # minimum drag for our L = W
# note that we didn't even have to make a constraint for L=W, since it is a definition

# note - why not minimize CD? CDA? aside about Rivians and smart cars

sol4 = opti.solve()

This is Ipopt version 3.14.11, running with linear solver MUMPS 5.4.1.

Number of nonzeros in equality constraint Jacobian...:        1
Number of nonzeros in inequality constraint Jacobian.:        8
Number of nonzeros in Lagrangian Hessian.............:       10

Total number of variables............................:        4
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Total number of equality constraints.................:        1
Total number of inequality constraints...............:        5
        inequality constraints with only lower bounds:        1
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:        4

iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
   0  5.7284779e+00 2.00e+00 4.65e+00   0.0 0.00e+00    -  0.00e+00 0.00e+00 

In [ ]:
AR_opt = sol4(AR)
S_opt = sol4(S)
b_opt = sol4(b)
D_opt = sol4(D)

print(f"AR* = {AR_opt} \nS* = {S_opt} m2 \nb* = {b_opt} m \nD* = {D_opt} N")

CL_opt = sol4(CL)
print(f"CL* = {CL_opt}")

CD_opt = sol4(CD)
print(f"CD* = {CD_opt}")

print(f"L/D * = {CL_opt/CD_opt}")

CD_i_opt = sol4(CD_i)
CD_v_opt = sol4(CD_v)
print(f"CDi* = {CD_i_opt} \nCDv* = {CD_v_opt}")

V_opt = sol4(airspeed)
print(f"V* = {V_opt} m/s")

Re_opt = sol4(Re)
print(f"Re* = {Re_opt}")

AR* = 10.000000098095231 
S* = 0.1037507641234706 m2 
b* = 1.0185811952967525 m 
D* = 0.12468735427171906 N
CL* = 0.6877957670669687
CD* = 0.031789244546332265
L/D * = 21.636115512733195
CDi* = 0.015058063368405318 
CDv* = 0.01673118117792695
V* = 7.856378577523679 m/s
Re* = 54159.614748788474


In [ ]:
# the result is the same, what gives?
# ASB gives us constraint sensitvities for free!
D_W_sens = sol4.value(
    opti.dual(W_sens_con)
)  # [N drag]/[N weight] (lagrange multiplier!)
print(D_W_sens)

# since this is a positive number, it means our drag D will increase when weight W is increased (assuming all else is fixed at the * point)

0.04378643756075123


In [ ]:
# we can easily translate this into L/D (range parameter)
dW = 1 * g / 1000  # [N/g] 1 gram of added mass == dW newtons of added weight
dD = D_W_sens * dW  # [N/g] drag increases by this many newtons

LD_start = sol4.value(W) / (D_opt)  # L/D *
LD_inc = (sol4.value(W) + dW) / (D_opt + dD)  # L/D with 1 extra gram
dLD = LD_inc - LD_start
print(f"delta L/D per gram of added mass: {dLD}")

# or, in % difference:
# note that you can also get percent change in an output by constraining the dummy variable to 1 and multiplying instead of adding it
pct = 100 * dLD / LD_start
print(f"percent change in L/D (range) per gram of added mass: {pct}")
# note that if the weight were increased by a factor of 1.74 (CLstall/CLstar), the plane wouldn't really fly anymore
# also note that the impact on sink rate (CL**(3/2)/CD) is worse, but we didn't optimize for that so whatever
# also, this is the power of MDO coupling -- if we can predict the weight of the glider, we can ensure it is the optimum design for that weight

delta L/D per gram of added mass: 0.004126667089963121
percent change in L/D (range) per gram of added mass: 0.01907304981587158


In [ ]:
# alternate method: L/D = W/D
# d/dW (W/D(W))
# quotient rule
# d(L/D)/dW = (D - W*dD/dW) / D^2 -> L/D change per Newton of added weight

(D_opt - sol4.value(W)*D_W_sens)/(D_opt**2) * dW

For most aircraft, it is better to spend a small amount of weight to reduce drag than to save a small about of weight by increasing drag.


But in our case, the aircraft wants to be heavier?? We can come back to this after adding a better weight model, but for now it seems like the glider wants some ballast.

In good lift conditions, sailplane pilots do add ballast to be able to fly faster at max L/D, but in this case, changing from drag minimization to L/D maximization (true max range) and adding a ballast variable is an ill-posed problem, due to the lack of structures coupling. Try it (predict what you think will happen first).

Exercise: sensitivities to different DVs? Some might be positive (span should be bigger) but are capped due to active constraints. If the lagrange multiplier is non-zero, the constraint is active.

In [ ]:
1 / D_W_sens  # very similar to L/D of 21.6